# 01 - Explore Iceberg Reference Data

Browse the 4 Iceberg tables that power the rightsizing engine:
- `rs_cloud_sku_catalog` - Instance types, specs, pricing
- `rs_cloud_sku_pricing_history` - CUR-derived effective prices
- `rs_finops_service_config` - 1,048+ service configs
- `rs_rightsizing_rules` - Sizing rules (19 generic + 26 Azure)

Also explores Bronze/Silver layer tables for debugging pipeline data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from local_helpers import athena_query, LocalIcebergReader
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

reader = LocalIcebergReader()
CLOUD = 'aws'  # Change to 'azure', 'gcp', 'oci', 'alibaba' as needed

## 1. SKU Catalog

In [ ]:
catalog_df = reader.load_sku_catalog(CLOUD)
print(f'Shape: {catalog_df.shape}')
print(f'Columns: {list(catalog_df.columns)}')
catalog_df.head(10)

In [ ]:
# Distribution by service_name
catalog_df['service_name'].value_counts().head(20)

In [ ]:
# Look up a specific SKU
sku = 'm5.xlarge'
catalog_df[catalog_df['sku_code'] == sku]

## 2. Pricing History (CUR-derived)

In [ ]:
pricing_df = reader.load_pricing_history(CLOUD, lookback_days=30)
print(f'Shape: {pricing_df.shape}')
print(f'Columns: {list(pricing_df.columns)}')
pricing_df.head(10)

In [ ]:
# Price trend for a specific SKU
sku = 'm5.xlarge'
sku_prices = pricing_df[pricing_df['sku_code'] == sku].sort_values('usage_date')
if not sku_prices.empty:
    print(f'Price history for {sku}:')
    display(sku_prices[['sku_code', 'region', 'effective_hourly_price', 'usage_date']].tail(10))
else:
    print(f'No pricing data for {sku}')

## 3. Service Configs

In [ ]:
configs_df = reader.load_service_configs(CLOUD)
print(f'Shape: {configs_df.shape}')
print(f'Columns: {list(configs_df.columns)}')
configs_df.head(20)

In [ ]:
# Service categories breakdown
configs_df['service_category'].value_counts()

## 4. Rightsizing Rules

In [ ]:
rules_df = reader.load_rules(CLOUD)
print(f'Shape: {rules_df.shape}')
print(f'Columns: {list(rules_df.columns)}')
rules_df

In [ ]:
# Rules by sizing method
if not rules_df.empty:
    rules_df.groupby(['service_category', 'sizing_method']).size().reset_index(name='count')

## 5. Browse Bronze / Silver Tables

In [ ]:
# List all Iceberg tables
db = os.getenv('ICEBERG_DB', 'finomics_catalog_data')
all_tables = athena_query(f'SHOW TABLES IN {db}')
all_tables

In [ ]:
# Sample from bronze EC2 instances (change table name as needed)
athena_query("""
    SELECT * FROM bronze_aws_ec2_instances
    LIMIT 5
""")

In [ ]:
# Sample from bronze CloudWatch metrics
athena_query("""
    SELECT * FROM bronze_aws_cloudwatch_metrics
    LIMIT 5
""")

In [ ]:
# Sample from silver recommendations
athena_query("""
    SELECT * FROM silver_aws_custom_recommendations
    LIMIT 10
""")

## 6. Ad-hoc Athena Queries

Use `athena_query()` for any custom SQL against Iceberg tables.

In [ ]:
# Your ad-hoc query here
athena_query("""
    SELECT cloud_provider, COUNT(*) as total_skus,
           COUNT(DISTINCT service_name) as services
    FROM rs_cloud_sku_catalog
    WHERE is_active = TRUE
    GROUP BY cloud_provider
    ORDER BY total_skus DESC
""")